# Drive and utilities


In [2]:
#@title Mount Drive to get BERT model
from google.colab import drive
drive.mount('/content/drive', force_remount=True)


Mounted at /content/drive


In [ ]:
#@title Utils
!pip install dill
!pip install pandas
!pip install numpy
!
import dill as pickle



In [ ]:
#@title Clone the repo and add API key
!git clone https://github.com/amusali/bert_for_patents.git
import os
# Api key setting
os.environ['patentsview_api_key'] = 'Uch9R1RR.BxaQcxV8HK4nBMgTTjX1F9CQW8PF4nXw'  # Replace with your actual key

In [ ]:
#@title Run setup file to get environment
#!python "/content/bert_for_patents/05 Analysis/01 Main/setup_colab.py"

import sys
sys.path.append('/content/bert_for_patents/05 Analysis/01 Main')
os.chdir("/content/bert_for_patents/05 Analysis/01 Main")

In [ ]:
#@title Pull latest changes from repo
%cd "/content/bert_for_patents"
!git pull origin master
#@title Add paths to the system and change directory
import sys
sys.path.append('/content/bert_for_patents/05 Analysis/01 Main')
os.chdir("/content/bert_for_patents/05 Analysis/01 Main")

# Calculate weighted before SMDs

In [ ]:
#@title Functions

import os, glob
import dill as pickle
import numpy as np
import pandas as pd

# ---------- 1) Point this to where you saved precomputed outputs ----------
# Example: "/content/drive/MyDrive/project/precomputed/*.pkl"

paths = sorted(glob.glob(PRECOMPUTED_GLOB, recursive=True))
print(f"Found {len(paths)} precomputed files")
for p in paths[:5]:
    print(" ", p)


# ---------- 2) Utilities ----------
def load_precomputed(path):
    """Load one precomputed object (list of dicts) saved with pickle.
    If you used dill, swap pickle for dill."""
    with open(path, "rb") as f:
        obj = pickle.load(f)
    return obj

def weighted_mean_and_var_from_sums(sum_w, sum_x, sum_x2):
    """Given weight sum and weighted sums of x and x^2, return mean and var."""
    mean = sum_x / sum_w
    var = (sum_x2 / sum_w) - mean**2
    # numerical floor
    var = np.maximum(var, 0.0)
    return mean, var

def compute_before_smds_from_precomputed(precomputed_list, last_k_quarters=4):
    """
    Compute BEFORE-matching SMDs for the last_k_quarters (default 4) closest to treatment:
      relative quarters: t-4, t-3, t-2, t-1

    precomputed_list: output of precompute_mahalanobis (list of dicts).
      Each dict contains:
        - 'treated_vectors': (n_t, Q)
        - 'candidate_vectors': (n_c, Q)
        - 'pre_quarters': list length Q (oldest -> most recent)
    """
    # We will aggregate across all risk sets with the "each treated contributes equally" weighting:
    # Treated: weight 1 per treated observation
    # Controls: within each treated risk set, each control gets weight 1/n_c, so total control weight per treated is 1.
    treated_w = 0.0
    control_w = 0.0

    treated_sum = None
    treated_sum2 = None
    control_sum = None
    control_sum2 = None

    used = 0
    skipped = 0

    for item in precomputed_list:
        T = np.asarray(item.get("treated_vectors"))
        C = np.asarray(item.get("candidate_vectors"))

        if T.ndim != 2 or C.ndim != 2:
            skipped += 1
            continue
        n_t, qT = T.shape
        n_c, qC = C.shape
        if n_t == 0 or n_c == 0 or qT != qC:
            skipped += 1
            continue

        Q = qT
        if last_k_quarters > Q:
            skipped += 1
            continue

        # slice to the last k quarters (closest to treatment)
        T4 = T[:, -last_k_quarters:]
        C4 = C[:, -last_k_quarters:]

        # initialize accumulators
        if treated_sum is None:
            treated_sum  = np.zeros(last_k_quarters, dtype=np.float64)
            treated_sum2 = np.zeros(last_k_quarters, dtype=np.float64)
            control_sum  = np.zeros(last_k_quarters, dtype=np.float64)
            control_sum2 = np.zeros(last_k_quarters, dtype=np.float64)

        # ---- Treated aggregation (unweighted per observation) ----
        treated_w   += n_t
        treated_sum += T4.sum(axis=0)
        treated_sum2 += (T4**2).sum(axis=0)

        # ---- Control aggregation with risk-set weighting ----
        # For each treated obs, controls have weights 1/n_c, so total weight per treated = 1.
        # Thus total control weight contributed by this item = n_t.
        control_w += n_t

        # For one treated obs, weighted mean over controls is mean(C4), and weighted E[X^2] is mean(C4^2)
        c_mean = C4.mean(axis=0)
        c_mean2 = (C4**2).mean(axis=0)

        control_sum  += n_t * c_mean
        control_sum2 += n_t * c_mean2

        used += 1

    if treated_sum is None:
        raise ValueError("No valid groups found in precomputed_list. Check your saved outputs.")

    mean_t, var_t = weighted_mean_and_var_from_sums(treated_w, treated_sum, treated_sum2)
    mean_c, var_c = weighted_mean_and_var_from_sums(control_w, control_sum, control_sum2)

    pooled_sd = np.sqrt((var_t + var_c) / 2.0)
    # avoid division by zero
    pooled_sd = np.where(pooled_sd == 0, np.nan, pooled_sd)

    smd = (mean_t - mean_c) / pooled_sd

    rel_q = [f"t-{last_k_quarters-i}" for i in range(last_k_quarters)]  # e.g., t-4, t-3, t-2, t-1

    out = pd.DataFrame({
        "rel_quarter": rel_q,
        "treated_mean": mean_t,
        "control_mean": mean_c,
        "treated_var": var_t,
        "control_var": var_c,
        "pooled_sd": pooled_sd,
        "SMD": smd,
        "abs_SMD": np.abs(smd),
    })

    meta = {
        "groups_used": used,
        "groups_skipped": skipped,
        "treated_weight_sum": treated_w,
        "control_weight_sum": control_w,
    }
    return out, meta


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Found 12 precomputed files
  /content/drive/MyDrive/uc3m PhD/PhD Data/11 Matches/actual results/citation_no_exact_match_on_grantyear/_aux/precomputed_mahalanobis_M&A_bl_4q.pkl
  /content/drive/MyDrive/uc3m PhD/PhD Data/11 Matches/actual results/citation_no_exact_match_on_grantyear/_aux/precomputed_mahalanobis_M&A_bl_6q.pkl
  /content/drive/MyDrive/uc3m PhD/PhD Data/11 Matches/actual results/citation_no_exact_match_on_grantyear/_aux/precomputed_mahalanobis_M&A_bl_8q.pkl
  /content/drive/MyDrive/uc3m PhD/PhD Data/11 Matches/actual results/citation_no_exact_match_on_grantyear/_aux/precomputed_mahalanobis_M&A_top_tech_80_4q.pkl
  /content/drive/MyDrive/uc3m PhD/PhD Data/11 Matches/actual results/citation_no_exact_match_on_grantyear/_aux/precomputed_mahalanobis_M&A_top_tech_80_6q.pkl


In [ ]:
#@title Calculate
all_tables = []
all_meta = []
from tqdm import tqdm

for path in tqdm(paths, total=len(paths), desc="Going over precomputes"):
    precomputed = load_precomputed(path)  # should be a list of dicts
    print("Loaded the file")
    table, meta = compute_before_smds_from_precomputed(precomputed, last_k_quarters=4)
    table.insert(0, "source_file", os.path.basename(path))
    all_tables.append(table)
    all_meta.append({"source_file": os.path.basename(path), **meta})

before_smds_df = pd.concat(all_tables, ignore_index=True)
meta_df = pd.DataFrame(all_meta)

print("Per-file SMDs (first rows):")
display(before_smds_df)

print("Meta diagnostics:")
display(meta_df)

Going over precomputes:   0%|          | 0/12 [00:00<?, ?it/s]

Loaded the file


Going over precomputes:   8%|▊         | 1/12 [01:21<14:56, 81.50s/it]

Loaded the file


Going over precomputes:  17%|█▋        | 2/12 [02:46<13:55, 83.56s/it]

...

Going over precomputes: 100%|██████████| 12/12 [09:48<00:00, 49.02s/it]

Per-file SMDs (first rows):


,source_file,rel_quarter,treated_mean,control_mean,treated_var,control_var,pooled_sd,SMD,abs_SMD
0,precomputed_mahalanobis_M&A_bl_4q.pkl,t-4,0.518160,0.306080,2.251144,1.772381,1.418366,0.149524,0.149524
1,precomputed_mahalanobis_M&A_bl_4q.pkl,t-3,0.541215,0.339206,3.420214,3.050589,1.798722,0.112307,0.112307
2,precomputed_mahalanobis_M&A_bl_4q.pkl,t-2,0.528898,0.326114,2.597623,2.105469,1.533475,0.132238,0.132238
3,precomputed_mahalanobis_M&A_bl_4q.pkl,t-1,0.480893,0.300985,2.467974,2.038165,1.501023,0.119857,0.119857
4,precomputed_mahalanobis_M&A_bl_6q.pkl,t-4,0.525424,0.311938,2.304743,1.809351,1.434241,0.148849,0.148849
5,precomputed_mahalanobis_M&A_bl_6q.pkl,t-3,0.549906,0.345223,3.520358,3.052346,1.812830,0.112908,0.112908
6,precomputed_mahalanobis_M&A_bl_6q.pkl,t-2,0.531184,0.332360,2.611274,2.125491,1.538955,0.129194,0.129194
7,precomputed_mahalanobis_M&A_bl_6q.pkl,t-1,0.486983,0.305698,2.542287,2.054412,1.516031,0.119579,0.119579
8,precomputed_mahalanobis_M&A_bl_8q.pkl,t-4,0.531123,0.318159,2.316280,1.845690,1.442562,0.147629,0.147629
9,precomputed_mahalanobis_M&A_bl_8q.pkl,t-3,0.552065,0.351530,3.570617,3.118823,1.828858,0.109651,0.109651


Meta diagnostics:


,source_file,groups_used,groups_skipped,treated_weight_sum,control_weight_sum
0,precomputed_mahalanobis_M&A_bl_4q.pkl,504,0,9499.0,9499.0
1,precomputed_mahalanobis_M&A_bl_6q.pkl,466,0,9027.0,9027.0
2,precomputed_mahalanobis_M&A_bl_8q.pkl,439,0,8595.0,8595.0
3,precomputed_mahalanobis_M&A_top_tech_80_4q.pkl,289,0,2604.0,2604.0
4,precomputed_mahalanobis_M&A_top_tech_80_6q.pkl,234,0,1861.0,1861.0
5,precomputed_mahalanobis_M&A_top_tech_80_8q.pkl,228,0,1940.0,1940.0
6,precomputed_mahalanobis_Off deal_bl_4q.pkl,305,0,6672.0,6672.0
7,precomputed_mahalanobis_Off deal_bl_6q.pkl,301,0,6333.0,6333.0
8,precomputed_mahalanobis_Off deal_bl_8q.pkl,296,0,5989.0,5989.0
9,precomputed_mahalanobis_Off deal_top_tech_80_4...,206,0,2526.0,2526.0


In [ ]:
#@title Save
# Save files
before_smds_df.to_excel(r"/content/drive/MyDrive/uc3m PhD/05 Analysis/01 Main/01 Stata/01 Main/04 Matching quality/out/00 Before SMDs - weeighted.xlsx")

# Calculate unweighted - stacked - before SMDs

In [ ]:
#@title Functions
import numpy as np
import pandas as pd

def compute_simple_before_smd(precomputed_list, last_k_quarters=4):

    treated_all = []
    control_all = []

    for item in precomputed_list:

        T = np.asarray(item.get("treated_vectors"))
        C = np.asarray(item.get("candidate_vectors"))

        if T.ndim != 2 or C.ndim != 2:
            continue

        if T.shape[1] < last_k_quarters:
            continue

        # take last k quarters (closest to treatment)
        treated_all.append(T[:, -last_k_quarters:])
        control_all.append(C[:, -last_k_quarters:])

    if not treated_all:
        raise ValueError("No valid groups found.")

    T_stack = np.vstack(treated_all)
    C_stack = np.vstack(control_all)

    results = []

    for q in range(last_k_quarters):

        treated_q = T_stack[:, q]
        control_q = C_stack[:, q]

        mean_t = treated_q.mean()
        mean_c = control_q.mean()

        sd_t = treated_q.std(ddof=1)
        sd_c = control_q.std(ddof=1)

        pooled_sd = np.sqrt((sd_t**2 + sd_c**2) / 2)

        smd = (mean_t - mean_c) / pooled_sd

        results.append({
            "quarter": f"t-{last_k_quarters-q}",
            "treated_mean": mean_t,
            "treated_sd": sd_t,
            "control_mean": mean_c,
            "control_sd": sd_c,
            "pooled_sd": pooled_sd,
            "SMD": smd,
            "abs_SMD": abs(smd)
        })

    return pd.DataFrame(results)

In [ ]:
#@title Calculate
all_tables_un = []
from tqdm import tqdm

for path in tqdm(paths, total=len(paths), desc="Going over precomputes"):
    precomputed = load_precomputed(path)  # should be a list of dicts
    print("Loaded the file")
    table = compute_simple_before_smd(precomputed, last_k_quarters=4)
    table.insert(0, "source_file", os.path.basename(path))
    all_tables_un.append(table)

before_smds_df_unweighted = pd.concat(all_tables_un, ignore_index=True)

print("Per-file SMDs (first rows):")
display(before_smds_df_unweighted)



Going over precomputes:   0%|          | 0/12 [00:00<?, ?it/s]

Loaded the file


Going over precomputes:   8%|▊         | 1/12 [01:28<16:10, 88.21s/it]

Loaded the file


Going over precomputes:  17%|█▋        | 2/12 [02:53<14:22, 86.26s/it]

...

Going over precomputes:  92%|█████████▏| 11/12 [10:20<00:43, 43.68s/it]

Loaded the file


Going over precomputes: 100%|██████████| 12/12 [10:41<00:00, 53.50s/it]

Per-file SMDs (first rows):


,source_file,quarter,treated_mean,treated_sd,control_mean,control_sd,pooled_sd,SMD,abs_SMD
0,precomputed_mahalanobis_M&A_bl_4q.pkl,t-4,0.518160,1.500460,0.301121,1.733219,1.621023,0.133890,0.133890
1,precomputed_mahalanobis_M&A_bl_4q.pkl,t-3,0.541215,1.849479,0.295333,1.645912,1.750657,0.140451,0.140451
2,precomputed_mahalanobis_M&A_bl_4q.pkl,t-2,0.528898,1.611799,0.296582,1.736611,1.675368,0.138666,0.138666
3,precomputed_mahalanobis_M&A_bl_4q.pkl,t-1,0.480893,1.571061,0.282669,1.528380,1.549868,0.127898,0.127898
4,precomputed_mahalanobis_M&A_bl_6q.pkl,t-4,0.525424,1.518222,0.300400,1.769565,1.648690,0.136486,0.136486
5,precomputed_mahalanobis_M&A_bl_6q.pkl,t-3,0.549906,1.876366,0.290677,1.641197,1.762708,0.147063,0.147063
6,precomputed_mahalanobis_M&A_bl_6q.pkl,t-2,0.531184,1.616033,0.292096,1.752568,1.685683,0.141835,0.141835
7,precomputed_mahalanobis_M&A_bl_6q.pkl,t-1,0.486983,1.594543,0.277766,1.540345,1.567678,0.133457,0.133457
8,precomputed_mahalanobis_M&A_bl_8q.pkl,t-4,0.531123,1.522021,0.303036,1.794464,1.663829,0.137086,0.137086
9,precomputed_mahalanobis_M&A_bl_8q.pkl,t-3,0.552065,1.889717,0.292601,1.668510,1.782548,0.145558,0.145558


In [ ]:
#@title Save
# Save files
before_smds_df_unweighted.to_excel(r"/content/drive/MyDrive/uc3m PhD/05 Analysis/01 Main/01 Stata/01 Main/04 Matching quality/out/00 Before SMDs - unweighted.xlsx")

# Before SMDs of PCAs

In [ ]:
# import a csv
t = pd.read_csv(r"/content/drive/MyDrive/uc3m PhD/PhD Data/01 CLS Embeddings/All embeddings - float16/PCA/pca_3D.csv")


/tmp/ipython-input-1154216605.py:2: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  t = pd.read_csv(r"/content/drive/MyDrive/uc3m PhD/PhD Data/01 CLS Embeddings/All embeddings - float16/PCA/pca_3D.csv")


In [ ]:
t.dtypes

,0
patent_id,object
PC1,float64
PC2,float64
PC3,float64


In [ ]:
#@title Functions
import os, glob
import dill as pickle
import numpy as np
import pandas as pd
PRECOMPUTED_GLOB = r"/content/drive/MyDrive/uc3m PhD/PhD Data/11 Matches/actual results/citation_no_exact_match_on_grantyear/_aux/precomputed*.pkl"

paths = sorted(glob.glob(PRECOMPUTED_GLOB, recursive=True))
print(f"Found {len(paths)} precomputed files")
PCA_CSV_PATH = r"/content/drive/MyDrive/uc3m PhD/PhD Data/01 CLS Embeddings/All embeddings - float16/PCA/pca_10D.csv"
N_PCS = 10

OUT_DIR = "/content/drive/MyDrive/uc3m PhD/05 Analysis/01 Main/01 Stata/01 Main/04 Matching quality/out"

# ----------------------------
# 1) Loaders
# ----------------------------

def load_precomputed(path):
    """Load one precomputed object (list of dicts) saved with pickle.
    If you used dill, swap pickle for dill."""
    with open(path, "rb") as f:
        obj = pickle.load(f)
    return obj


def load_pca_csv(path: str, n_pcs: int = 10) -> pd.DataFrame:
    # Force patent_id to string to avoid mixed-type warnings
    df = pd.read_csv(path, dtype={"patent_id": "string"}, low_memory=False)

    required = ["patent_id"] + [f"PC{i}" for i in range(1, n_pcs + 1)]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(
            f"PCA file missing columns: {missing}\n"
            f"Found columns (first 40): {df.columns.tolist()[:40]}"
        )

    for i in range(1, n_pcs + 1):
        df[f"PC{i}"] = pd.to_numeric(df[f"PC{i}"], errors="coerce")

    df = df.dropna(subset=["patent_id"])
    df["patent_id"] = df["patent_id"].astype("string")

    # Keep only required columns
    return df[required]


# ----------------------------
# 2) Extract UNIQUE treated ids and UNIQUE candidate ids
# ----------------------------
def extract_unique_ids(precomputed_list):
    treated_ids = set()
    candidate_ids = set()

    for item in precomputed_list:
        tids = item.get("treated_ids", [])
        cids = item.get("candidate_ids", [])

        # normalize to string
        for t in tids:
            treated_ids.add(str(t))
        for c in cids:
            candidate_ids.add(str(c))

    return treated_ids, candidate_ids


# ----------------------------
# 3) SMD computation (unweighted, two unique samples)
# ----------------------------
def smd_table_unweighted(treated_df: pd.DataFrame, control_df: pd.DataFrame, covariates):
    rows = []
    for var in covariates:
        xt = treated_df[var].to_numpy(dtype=np.float64)
        xc = control_df[var].to_numpy(dtype=np.float64)

        # drop NaN (just in case)
        xt = xt[np.isfinite(xt)]
        xc = xc[np.isfinite(xc)]

        mean_t = xt.mean()
        mean_c = xc.mean()
        sd_t = xt.std(ddof=1)
        sd_c = xc.std(ddof=1)
        pooled_sd = np.sqrt((sd_t**2 + sd_c**2) / 2.0)
        smd = (mean_t - mean_c) / pooled_sd if pooled_sd != 0 else np.nan

        rows.append({
            "covariate": var,
            "treated_mean": mean_t,
            "treated_sd": sd_t,
            "control_mean": mean_c,
            "control_sd": sd_c,
            "pooled_sd": pooled_sd,
            "SMD": smd,
            "abs_SMD": abs(smd) if np.isfinite(smd) else np.nan,
            "n_treated": len(xt),
            "n_controls": len(xc),
        })
    return pd.DataFrame(rows)


Found 12 precomputed files


In [ ]:
#@title Calculate
from tqdm import tqdm

# Load PCA
pca = load_pca_csv(PCA_CSV_PATH, n_pcs=N_PCS)
print("PCA loaded:", pca.shape)

# Load all precomputed files
paths = sorted(glob.glob(PRECOMPUTED_GLOB, recursive=True))
if not paths:
    raise FileNotFoundError(f"No precomputed files found with: {PRECOMPUTED_GLOB}")

all_results = []

for p in tqdm(paths, total = len(paths), desc = "Loopoing over paths"):
    precomputed = load_precomputed(p)
    if not isinstance(precomputed, list):
        raise TypeError(f"Expected list in {p}, got {type(precomputed)}")

    # unique IDs *within this file/config only*
    treated_ids, candidate_ids = extract_unique_ids(precomputed)

    print(f"number of controls before: {len(candidate_ids)}")
    # IMPORTANT:
    candidate_ids = candidate_ids - treated_ids  # keep sets disjoint
    print(f"number of controls after: {len(candidate_ids)}")

    treated_pca = pca[pca["patent_id"].isin(treated_ids)].copy()
    control_pca = pca[pca["patent_id"].isin(candidate_ids)].copy()

    pcs = [f"PC{i}" for i in range(1, N_PCS + 1)]
    balance_pcs = smd_table_unweighted(treated_pca, control_pca, pcs)

    # tag results with config/file name
    balance_pcs.insert(0, "config_file", os.path.basename(p))
    balance_pcs.insert(1, "n_unique_treated_ids", len(treated_ids))
    balance_pcs.insert(2, "n_unique_candidate_ids", len(candidate_ids))
    balance_pcs.insert(3, "n_treated_with_pca", len(treated_pca))
    balance_pcs.insert(4, "n_controls_with_pca", len(control_pca))

    all_results.append(balance_pcs)

# --- CHANGE 2: combine results at the very end (no cross-file dedupe/stacking) ---
results_df = pd.concat(all_results, ignore_index=True)

Loopoing over paths:   0%|          | 0/12 [00:00<?, ?it/s]

number of controls before: 4067852
number of controls after: 4067852


Loopoing over paths:   8%|▊         | 1/12 [02:23<26:15, 143.26s/it]

number of controls before: 3925937
number of controls after: 3925937


Loopoing over paths:  17%|█▋        | 2/12 [04:46<23:52, 143.22s/it]

...

Loopoing over paths:  83%|████████▎ | 10/12 [15:07<02:32, 76.17s/it]

number of controls before: 1884303
number of controls after: 1884303


Loopoing over paths:  92%|█████████▏| 11/12 [15:44<01:04, 64.14s/it]

number of controls before: 1872125
number of controls after: 1872125


Loopoing over paths: 100%|██████████| 12/12 [16:26<00:00, 82.23s/it]


In [ ]:
# --- CHANGE 3: save single output (or one-per-file if you prefer) ---
out_path = os.path.join(OUT_DIR, "00 Before SMDs PCAs.csv")
results_df.to_csv(out_path, index=False)
print("Saved:", out_path)

Saved: /content/drive/MyDrive/uc3m PhD/05 Analysis/01 Main/01 Stata/01 Main/04 Matching quality/out/00 Before SMDs PCAs.csv


In [ ]:
from google.colab import sheets
sheet = sheets.InteractiveSheet(df=results_df)

MessageError: Error: credential propagation was unsuccessful

In [3]:
import pandas as pd

pca = pd.read_csv(r"/content/drive/MyDrive/uc3m PhD/PhD Data/01 CLS Embeddings/All embeddings - float16/PCA/pca_10D.csv")

/tmp/ipython-input-1455/3290459783.py:3: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  pca = pd.read_csv(r"/content/drive/MyDrive/uc3m PhD/PhD Data/01 CLS Embeddings/All embeddings - float16/PCA/pca_10D.csv")


In [4]:
pca.head()

,patent_id,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10
0,10000001,0.866372,5.954933,6.394658,-0.551546,4.521981,-0.101108,1.240542,-5.135097,-1.519952,-3.057474
1,10000002,1.121164,6.867592,2.962751,3.998940,-1.723198,-6.093444,-3.118760,1.807858,-3.117344,0.631172
2,10000003,3.121801,3.129339,-2.616843,4.916008,-0.811856,-7.222512,-0.979107,2.071569,-3.873968,-5.267483
3,10000004,0.534734,-0.204596,2.541621,-4.801057,0.431866,-3.266522,-2.002393,2.284087,-7.102862,-0.468413
4,10000005,2.865330,3.024400,0.713367,1.831444,3.363750,-5.766662,1.591319,0.632718,-1.417394,-1.508772


In [12]:
print(pca[pca['patent_id'] == 10078376])

        patent_id       PC1       PC2       PC3       PC4       PC5       PC6  \
7583260  10078376  3.488712  7.801583 -5.674872 -0.011894 -2.200841  0.082953   

              PC7       PC8       PC9      PC10  
7583260 -0.122243 -1.447685 -1.802509  1.557432  


In [ ]:
wr